# SafeSite Vision — Training on Kaggle

This notebook:
1. Installs dependencies
2. Locates the dataset (Kaggle input or uploaded)
3. Runs `scripts/train.py` for one experiment
4. Saves `best.pt` and `reports/experiments.csv` as Kaggle output

**Run on a Kaggle GPU accelerator (T4 x2 or P100).**

In [ ]:
# ── 1. Install / upgrade dependencies ────────────────────────────────────────
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'ultralytics', 'pyyaml'], check=True)
print('Dependencies ready.')

In [ ]:
# ── 2. Locate the dataset ─────────────────────────────────────────────────────
from pathlib import Path

# Kaggle input datasets are mounted at /kaggle/input/<dataset-slug>/
KAGGLE_INPUT = Path('/kaggle/input')
WORKING      = Path('/kaggle/working')

# Expected: the processed dataset was uploaded as a Kaggle dataset
# or generated by prepare_dataset.py in a prior notebook.
CANDIDATES = [
    KAGGLE_INPUT / 'safesite-processed' / 'data.yaml',
    WORKING / 'data' / 'processed' / 'data.yaml',
]

DATA_YAML = next((p for p in CANDIDATES if p.exists()), None)
if DATA_YAML is None:
    raise FileNotFoundError(
        'data.yaml not found. Upload the processed dataset to Kaggle or '
        'run prepare_dataset.py first.\nChecked:\n' +
        '\n'.join(str(p) for p in CANDIDATES)
    )

print(f'Using dataset: {DATA_YAML}')

In [ ]:
# ── 3. Clone the repo (if not already present) ───────────────────────────────
import os

REPO_DIR = WORKING / 'safesite-vision'

if not REPO_DIR.exists():
    # Replace with your actual GitHub URL after pushing
    GITHUB_URL = 'https://github.com/shabanhassan142/safesite-vision.git'
    subprocess.run(['git', 'clone', '--depth', '1', GITHUB_URL, str(REPO_DIR)], check=True)
else:
    print('Repo already present, pulling latest...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)

os.chdir(REPO_DIR)
print(f'Working directory: {Path.cwd()}')

In [ ]:
# ── 4. Write a Kaggle-specific train.yaml override ───────────────────────────
import yaml

base_cfg_path = REPO_DIR / 'configs' / 'train.yaml'
with open(base_cfg_path) as f:
    cfg = yaml.safe_load(f)

# Override paths and batch for Kaggle GPU
cfg['data']    = str(DATA_YAML)
cfg['project'] = str(WORKING / 'models')
cfg['batch']   = 16   # T4 can handle 16 at imgsz 640
cfg['workers'] = 4

# Remove device override — Ultralytics auto-detects GPU on Kaggle
# (build_args sets device='cpu' for local runs; on Kaggle we want GPU)
kaggle_cfg_path = WORKING / 'train_kaggle.yaml'
with open(kaggle_cfg_path, 'w') as f:
    yaml.dump(cfg, f)

print(f'Kaggle config written to {kaggle_cfg_path}')
print(yaml.dump(cfg))

In [ ]:
# ── 5. Run training (GPU — device override happens inside train.py) ───────────
# We patch build_args to use device='0' (first GPU) on Kaggle.
# The simplest way: set CUDA_VISIBLE_DEVICES and let Ultralytics auto-detect.

import sys
sys.path.insert(0, str(REPO_DIR / 'scripts'))

import train as train_module
from train import load_config, build_args, run_training

# Monkey-patch build_args to use GPU on Kaggle
_orig_build_args = build_args
def _kaggle_build_args(cfg):
    args = _orig_build_args(cfg)
    args['device'] = '0'   # first GPU
    return args
train_module.build_args = _kaggle_build_args

cfg = load_config(kaggle_cfg_path)
print(f'Experiment: {cfg.experiment_id}  model: {cfg.model}  '
      f'imgsz: {cfg.imgsz}  epochs: {cfg.epochs}')

results = run_training(cfg)

In [ ]:
# ── 6. Verify outputs and copy to /kaggle/working ────────────────────────────
import shutil

models_dir  = Path(cfg.project) / cfg.name
weights_dir = models_dir / 'weights'

print('Files in run directory:')
for f in sorted(models_dir.rglob('*')):
    print(' ', f.relative_to(models_dir))

# Make sure best.pt is at the top level of /kaggle/working for easy download
best_src = weights_dir / 'best.pt'
if best_src.exists():
    shutil.copy2(best_src, WORKING / f'best_{cfg.experiment_id}.pt')
    print(f'\nbest.pt copied to {WORKING / ("best_" + cfg.experiment_id + ".pt")}')
else:
    print('WARNING: best.pt not found — check training output above')

# Copy experiments.csv
exp_csv = REPO_DIR / 'reports' / 'experiments.csv'
if exp_csv.exists():
    shutil.copy2(exp_csv, WORKING / 'experiments.csv')
    print(f'experiments.csv copied to {WORKING / "experiments.csv"}')

In [ ]:
# ── 7. Quick summary ─────────────────────────────────────────────────────────
import csv as csv_mod

exp_csv_working = WORKING / 'experiments.csv'
if exp_csv_working.exists():
    rows = list(csv_mod.DictReader(open(exp_csv_working)))
    print('\nExperiment log:')
    for r in rows:
        print(f"  [{r['experiment_id']}] model={r['model']} imgsz={r['imgsz']} "
              f"mAP50={r['map50']} precision={r['precision']} "
              f"recall={r['recall']} time={r['train_time_s']}s")
else:
    print('experiments.csv not found — training may have failed')